# Chapter 1: Foundations of Agentic AI
## Code Examples and Implementations

This notebook contains practical implementations of the concepts covered in Chapter 1:
- Basic agent structures
- ReAct pattern implementation
- Chain-of-Thought prompting
- Framework comparisons with working examples

### Setup

First, ensure you have the required dependencies installed using `uv`:

```bash
uv add anthropic
uv add openai
uv add langchain
uv add langchain-openai
uv add langgraph
uv add crewai
uv add python-dotenv
```

In [ ]:
# Import required libraries
import os
from dotenv import load_dotenv
import json
from typing import List, Dict, Any, Callable

# Load environment variables (API keys)
load_dotenv()

# Verify API keys are loaded
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found in environment"
print("Environment setup complete")

## Part 1: Building a Simple ReAct Agent from Scratch

Let's implement the ReAct pattern to understand how the thought-action-observation loop works.

In [ ]:
from openai import OpenAI

class SimpleReActAgent:
    """A simple implementation of the ReAct pattern."""
    
    def __init__(self, model: str = "gpt-4"):
        self.client = OpenAI()
        self.model = model
        self.tools = {}
        self.max_iterations = 10
        
    def register_tool(self, name: str, func: Callable, description: str):
        """Register a tool that the agent can use."""
        self.tools[name] = {
            "function": func,
            "description": description
        }
        
    def _format_tools(self) -> str:
        """Format available tools for the prompt."""
        tool_descriptions = []
        for name, tool in self.tools.items():
            tool_descriptions.append(f"- {name}: {tool['description']}")
        return "\n".join(tool_descriptions)
    
    def run(self, query: str) -> str:
        """Execute the ReAct loop."""
        conversation_history = []
        
        system_prompt = f"""You are a helpful AI agent that uses the ReAct pattern.
For each step, you should:
1. Think about what to do next (Thought)
2. Decide on an action to take (Action)
3. Observe the result (Observation)

Available tools:
{self._format_tools()}

Format your response as:
Thought: [your reasoning]
Action: [tool_name(arguments)] OR Final Answer: [answer if done]

Continue this loop until you can provide a Final Answer."""
        
        conversation_history.append({
            "role": "system",
            "content": system_prompt
        })
        
        conversation_history.append({
            "role": "user",
            "content": query
        })
        
        for iteration in range(self.max_iterations):
            # Get agent's response
            response = self.client.chat.completions.create(
                model=self.model,
                messages=conversation_history,
                temperature=0
            )
            
            agent_response = response.choices[0].message.content
            print(f"\n=== Iteration {iteration + 1} ===")
            print(agent_response)
            
            # Check if we have a final answer
            if "Final Answer:" in agent_response:
                final_answer = agent_response.split("Final Answer:")[1].strip()
                return final_answer
            
            # Parse and execute action
            if "Action:" in agent_response:
                action_line = [line for line in agent_response.split("\n") 
                             if line.startswith("Action:")][0]
                action = action_line.replace("Action:", "").strip()
                
                # Execute the tool
                observation = self._execute_action(action)
                
                print(f"\nObservation: {observation}")
                
                # Add to conversation history
                conversation_history.append({
                    "role": "assistant",
                    "content": agent_response
                })
                conversation_history.append({
                    "role": "user",
                    "content": f"Observation: {observation}"
                })
            else:
                conversation_history.append({
                    "role": "assistant",
                    "content": agent_response
                })
        
        return "Maximum iterations reached without finding answer"
    
    def _execute_action(self, action: str) -> str:
        """Execute a tool action and return the observation."""
        try:
            # Parse tool name and arguments
            if "(" in action:
                tool_name = action.split("(")[0].strip()
                args_str = action.split("(")[1].split(")")[0]
                
                if tool_name in self.tools:
                    # Execute tool
                    result = self.tools[tool_name]["function"](args_str)
                    return str(result)
                else:
                    return f"Error: Tool '{tool_name}' not found"
            else:
                return "Error: Invalid action format"
        except Exception as e:
            return f"Error executing action: {str(e)}"

### Testing the ReAct Agent

Let's create some simple tools and test the agent.

In [ ]:
# Define simple tools
def calculator(expression: str) -> float:
    """Evaluate a mathematical expression."""
    try:
        # Only allow safe operations
        allowed_chars = set("0123456789+-*/(). ")
        if not all(c in allowed_chars for c in expression):
            return "Error: Invalid characters in expression"
        result = eval(expression)
        return result
    except Exception as e:
        return f"Error: {str(e)}"

def get_current_date(args: str) -> str:
    """Get the current date."""
    from datetime import datetime
    return datetime.now().strftime("%Y-%m-%d")

# Create and configure agent
agent = SimpleReActAgent(model="gpt-4")
agent.register_tool(
    "calculator",
    calculator,
    "Evaluate mathematical expressions. Format: calculator(expression)"
)
agent.register_tool(
    "get_current_date",
    get_current_date,
    "Get the current date in YYYY-MM-DD format. Format: get_current_date()"
)

# Test the agent
result = agent.run(
    "If I invest $1000 at 5% annual interest, how much will I have after 3 years with compound interest?"
)

print(f"\n\n=== FINAL RESULT ===")
print(result)

## Part 2: Chain-of-Thought Prompting

Demonstrating the difference between regular prompting and CoT prompting.

In [ ]:
from openai import OpenAI

client = OpenAI()

# Test problem
problem = """A farmer has 17 sheep. All but 9 die. How many sheep are left?"""

# Without Chain-of-Thought
print("=== WITHOUT Chain-of-Thought ===")
response_no_cot = client.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role": "user", "content": problem}
    ],
    temperature=0
)
print(response_no_cot.choices[0].message.content)

# With Chain-of-Thought
print("\n=== WITH Chain-of-Thought ===")
response_with_cot = client.chat.completions.create(
    model="gpt-4",
    messages=[
        {
            "role": "user",
            "content": f"{problem}\n\nPlease think through this step-by-step before providing your final answer."
        }
    ],
    temperature=0
)
print(response_with_cot.choices[0].message.content)

## Part 3: LangChain Agent Example

Building an agent using LangChain's framework.

In [ ]:
from langchain.agents import Tool, AgentExecutor, create_react_agent
from langchain_openai import ChatOpenAI
from langchain import hub

# Define tools
def multiply(a: str) -> str:
    """Multiply two numbers separated by comma."""
    try:
        nums = [float(x.strip()) for x in a.split(",")]
        if len(nums) != 2:
            return "Error: Please provide exactly two numbers separated by comma"
        return str(nums[0] * nums[1])
    except Exception as e:
        return f"Error: {str(e)}"

def add(a: str) -> str:
    """Add two numbers separated by comma."""
    try:
        nums = [float(x.strip()) for x in a.split(",")]
        if len(nums) != 2:
            return "Error: Please provide exactly two numbers separated by comma"
        return str(nums[0] + nums[1])
    except Exception as e:
        return f"Error: {str(e)}"

tools = [
    Tool(
        name="Multiply",
        func=multiply,
        description="Multiply two numbers. Input should be two numbers separated by comma, like '5, 3'"
    ),
    Tool(
        name="Add",
        func=add,
        description="Add two numbers. Input should be two numbers separated by comma, like '5, 3'"
    )
]

# Initialize LLM
llm = ChatOpenAI(model="gpt-4", temperature=0)

# Get ReAct prompt from hub
prompt = hub.pull("hwchase17/react")

# Create agent
agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True
)

# Run agent
result = agent_executor.invoke({
    "input": "What is (5 * 3) + 7?"
})

print(f"\nFinal Answer: {result['output']}")

## Part 4: CrewAI Multi-Agent Example

Demonstrating role-based agent collaboration with CrewAI.

In [ ]:
from crewai import Agent, Task, Crew, Process
from langchain_openai import ChatOpenAI

# Initialize LLM for agents
llm = ChatOpenAI(model="gpt-4")

# Define agents with specific roles
researcher = Agent(
    role='Data Analyst',
    goal='Analyze data and extract insights',
    backstory="""You are an expert data analyst with years of experience
    in finding patterns and extracting meaningful insights from data.""",
    llm=llm,
    verbose=True
)

writer = Agent(
    role='Technical Writer',
    goal='Create clear, concise summaries of technical findings',
    backstory="""You are a skilled technical writer who can explain
    complex concepts in simple, understandable terms.""",
    llm=llm,
    verbose=True
)

# Define tasks
analysis_task = Task(
    description="""Analyze the following scenario: A company's monthly revenue
    has been: Jan: $100k, Feb: $120k, Mar: $115k, Apr: $140k, May: $138k.
    Identify trends and provide insights.""",
    agent=researcher,
    expected_output="A detailed analysis of revenue trends"
)

writing_task = Task(
    description="""Based on the analysis, write a brief executive summary
    that highlights the key findings in 3-4 sentences.""",
    agent=writer,
    expected_output="A concise executive summary",
    context=[analysis_task]  # This task depends on analysis_task
)

# Create crew
crew = Crew(
    agents=[researcher, writer],
    tasks=[analysis_task, writing_task],
    process=Process.sequential,  # Tasks run in sequence
    verbose=True
)

# Execute crew
result = crew.kickoff()

print(f"\n=== FINAL RESULT ===")
print(result)

## Part 5: Comparing Agent Responses Across Frameworks

Let's compare how different frameworks handle the same task.

In [ ]:
test_query = "Calculate the area of a circle with radius 5, then multiply by 2"

print("=== Testing the same query across different implementations ===")
print(f"Query: {test_query}\n")

# 1. Simple ReAct Agent
print("\n1. SIMPLE REACT AGENT")
print("-" * 50)
simple_agent = SimpleReActAgent()
simple_agent.register_tool(
    "calculator",
    calculator,
    "Evaluate mathematical expressions including pi constant"
)
simple_result = simple_agent.run(test_query)
print(f"Result: {simple_result}")

# 2. LangChain Agent
print("\n2. LANGCHAIN AGENT")
print("-" * 50)
# (Would use the LangChain setup from earlier)
print("Result: (See LangChain example above)")

# 3. Direct LLM with CoT
print("\n3. DIRECT LLM WITH CHAIN-OF-THOUGHT")
print("-" * 50)
response = client.chat.completions.create(
    model="gpt-4",
    messages=[
        {
            "role": "user",
            "content": f"{test_query}\n\nPlease solve this step-by-step."
        }
    ],
    temperature=0
)
print(response.choices[0].message.content)

## Part 6: Understanding Agent State and Memory

A simple demonstration of how agents maintain state.

In [ ]:
class StatefulAgent:
    """An agent that maintains conversation state."""
    
    def __init__(self, model: str = "gpt-4"):
        self.client = OpenAI()
        self.model = model
        self.conversation_history = []
        self.memory = {}  # Key-value store for facts
        
    def remember(self, key: str, value: Any):
        """Store a fact in memory."""
        self.memory[key] = value
        print(f"Remembered: {key} = {value}")
        
    def recall(self, key: str) -> Any:
        """Retrieve a fact from memory."""
        return self.memory.get(key)
    
    def chat(self, message: str) -> str:
        """Have a conversation with the agent."""
        # Add user message to history
        self.conversation_history.append({
            "role": "user",
            "content": message
        })
        
        # Create system message with memory context
        system_message = "You are a helpful assistant."
        if self.memory:
            memory_context = "\n".join(
                f"- {k}: {v}" for k, v in self.memory.items()
            )
            system_message += f"\n\nKnown facts:\n{memory_context}"
        
        messages = [
            {"role": "system", "content": system_message}
        ] + self.conversation_history
        
        # Get response
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=0.7
        )
        
        assistant_message = response.choices[0].message.content
        
        # Add to history
        self.conversation_history.append({
            "role": "assistant",
            "content": assistant_message
        })
        
        return assistant_message

# Test stateful agent
agent = StatefulAgent()

# Store some facts
agent.remember("user_name", "Alice")
agent.remember("user_role", "Data Scientist")
agent.remember("project", "Customer Churn Prediction")

# Have a conversation
print("\n=== Conversation ===")
response1 = agent.chat("What project am I working on?")
print(f"Agent: {response1}")

response2 = agent.chat("What's my role?")
print(f"\nAgent: {response2}")

response3 = agent.chat("Can you summarize what you know about me?")
print(f"\nAgent: {response3}")

## Summary

In this notebook, we implemented:

1. **Simple ReAct Agent**: A from-scratch implementation of the thought-action-observation loop
2. **Chain-of-Thought**: Demonstrations of improved reasoning with step-by-step thinking
3. **LangChain Agent**: Using a production framework for agent development
4. **CrewAI Multi-Agent**: Role-based agent collaboration
5. **Framework Comparison**: Testing the same task across different approaches
6. **Stateful Agent**: Managing conversation history and persistent memory

### Key Takeaways:

- The ReAct pattern provides a reliable structure for agent reasoning
- Chain-of-Thought prompting improves reasoning quality
- Different frameworks offer trade-offs in ease of use vs control
- State management is crucial for multi-turn agent interactions
- Production agents require robust error handling and observability

### Next Steps:

- Explore more complex tool integrations
- Implement custom memory systems (covered in Chapter 2)
- Build multi-agent workflows with specialized agents
- Add proper logging and observability
- Deploy agents in production environments